# 14 — Trade Execution

## Goal
Translate a confirmed, scored zone into **entry, stop-loss, and take-profit** levels.

---

## Geometry

### Demand zone (buy)

$$\text{entry} = \text{proximal}$$
$$\text{sl}    = \text{distal} - 0.1 \times \overline{\text{ATR}}$$
$$\text{risk}  = \text{entry} - \text{sl}$$
$$\text{tp}    = \text{entry} + 3 \times \text{risk}$$

### Supply zone (sell)

$$\text{entry} = \text{proximal}$$
$$\text{sl}    = \text{distal} + 0.1 \times \overline{\text{ATR}}$$
$$\text{risk}  = \text{sl} - \text{entry}$$
$$\text{tp}    = \text{entry} - 3 \times \text{risk}$$

The $0.1 \times \text{ATR}$ buffer absorbs typical stop-hunt wicks without giving up too much R.  
The **1:3 risk-reward** is the minimum; anything below is skipped regardless of SETS score.

---

## R:R Gate

Only take the trade if $\text{risk} > 0$ (non-degenerate zone) and the calculated TP is meaningful.

---

## Visualization
For each zone, entry / SL / TP levels drawn as horizontal lines on the chart.  
Green lines = demand trade, Red lines = supply trade.


## 1. Load data and run the detection pipeline

In [4]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


9 zones detected


## 2. `compute_trade_levels` for each zone

In [5]:
ATR_BUFFER = 0.10

def compute_trade_levels(zone):
    avg_atr = atr_arr[zone["bs"]:zone["be"]+1].mean()
    zt      = zone["zone_type"]
    prox    = zone["proximal"]
    dist    = zone["distal"]
    if zt == "demand":
        entry = prox
        sl    = dist - ATR_BUFFER * avg_atr
        risk  = entry - sl
        tp    = entry + 3 * risk
    else:
        entry = prox
        sl    = dist + ATR_BUFFER * avg_atr
        risk  = sl - entry
        tp    = entry - 3 * risk
    rr_ok = risk > 0
    return dict(entry=round(entry,3), sl=round(sl,3),
                tp=round(tp,3), risk=round(risk,3),
                rr_ok=rr_ok)

for z in zones:
    z.update(compute_trade_levels(z))
    z["scenario"] = labeled["scenario"].iloc[z["bs"]]

print(f"{'Scenario':<22} {'Type':8} {'Entry':>7} {'SL':>7} {'TP':>7} {'Risk':>6} {'R:R'}")
print("-" * 72)
for z in zones:
    rr_str = "1:3 ✓" if z["rr_ok"] else "skip"
    print(f"{z['scenario']:<22} {z['zone_type']:8} {z['entry']:>7.2f} "
          f"{z['sl']:>7.2f} {z['tp']:>7.2f} {z['risk']:>6.3f} {rr_str}")


Scenario               Type       Entry      SL      TP   Risk R:R
------------------------------------------------------------------------
A_demand_RBR           demand    105.90  104.74  109.38  1.159 1:3 ✓
A_demand_RBR           supply    109.00  110.57  104.28  1.574 1:3 ✓
B_supply_DBD           supply    105.80  106.88  102.57  1.076 1:3 ✓
D_wide_base            demand    115.80  114.18  120.67  1.623 1:3 ✓
E_doji_base            demand    120.90  119.79  124.24  1.114 1:3 ✓
E_doji_base            demand    125.40  123.78  130.25  1.617 1:3 ✓
H_DBR                  demand    135.70  134.49  139.34  1.213 1:3 ✓
H_DBR                  demand    140.10  138.78  144.06  1.320 1:3 ✓
I_RBD                  supply    144.60  145.81  140.96  1.215 1:3 ✓


## 3. Visualization — entry / SL / TP on the chart

In [6]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="Price",
))

for z in zones:
    if not z["rr_ok"]: continue
    x0   = df.index[z["bs"]]
    xend = df.index[min(z["be"] + DEPARTURE_CANDLES + 5, len(df)-1)]
    c    = EDGE[z["zone_type"]]
    # entry
    fig.add_shape(type="line", x0=x0, x1=xend,
                  y0=z["entry"], y1=z["entry"],
                  line=dict(color=c, width=1.5, dash="dash"))
    # SL
    fig.add_shape(type="line", x0=x0, x1=xend,
                  y0=z["sl"], y1=z["sl"],
                  line=dict(color="#ef5350", width=1, dash="dot"))
    # TP
    fig.add_shape(type="line", x0=x0, x1=xend,
                  y0=z["tp"], y1=z["tp"],
                  line=dict(color="#26a69a", width=1, dash="dot"))
    fig.add_annotation(x=xend, y=z["entry"],
                       text=f"{z['formation']} E={z['entry']:.2f}",
                       showarrow=False, font=dict(size=8, color=c),
                       xanchor="left")

fig.update_layout(
    title="Trade Execution — Entry / SL / TP per zone",
    height=540, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)
fig.show()
